# LOSO (df4만 사용)

In [ ]:

X_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_filtered_cutoff0.005.csv", index_col=0)


meta_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/meta_df4_filtered_cutoff0.005.csv", index_col=0)
batch_filtered_df4 =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/batch_df4_filtered_cutoff0.005.csv", index_col=0)

X_df4_latgentgee =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/results/df4/phase2/X_corrected_df4_clean_latentgee.csv", index_col=0)

X_df4_combat =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_combat.csv", index_col=0)
X_df4_mmuphine =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_MMUPHin.csv", index_col=0)
X_df4_conqur =pd.read_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/data/X_df4_corrected_ConQuR.csv", index_col=0)


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
def loso_cv(X, y, batch, label=""):
    # 먼저 array로 변환
    X = np.array(X)
    y = np.array(y)
    batch = np.array(batch)
    
    # hivstatus NaN 제거
    valid_mask = ~pd.isna(y)
    X = X[valid_mask]
    y = y[valid_mask]
    batch = batch[valid_mask]
    
    # label encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    studies = np.unique(batch)
    aucs = []
    
    for study in studies:
        test_mask = batch == study
        train_mask = ~test_mask
        
        X_train = X[train_mask]
        X_test = X[test_mask]
        y_train = y_encoded[train_mask]
        y_test = y_encoded[test_mask]
        
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            continue
        
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
        clf.fit(X_train, y_train)
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        aucs.append({"study": study, "auc": auc})
    
    df_auc = pd.DataFrame(aucs)
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"Mean AUC: {df_auc['auc'].mean():.4f} ± {df_auc['auc'].std():.4f}")
    print(df_auc.to_string(index=False))
    
    return df_auc

In [30]:
# df4 기준 LOSO
bio_labels = meta_filtered_df4['hivstatus'].values
# Before
auc_before = loso_cv(
    X_filtered_df4.values,
    bio_labels,
    batch_filtered_df4['Study'].values,
    label="df4 — Before"
)

# LatentGEE
auc_latentgee = loso_cv(
    X_df4_latgentgee.values,
    bio_labels,
    batch_filtered_df4['Study'].values,
    label="df4 — LatentGEE"
)

# ComBat
auc_combat = loso_cv(
    X_df4_combat.values,
    bio_labels,
    batch_filtered_df4['Study'].values,
    label="df4 — ComBat"
)

# MMUPHin
auc_mmuphin = loso_cv(
    X_df4_mmuphine.values,
    bio_labels,
    batch_filtered_df4['Study'].values,
    label="df4 — MMUPHin"
)

# ConQuR
auc_conqur = loso_cv(
    X_df4_conqur.values,
    bio_labels,
    batch_filtered_df4['Study'].values,
    label="df4 — ConQuR"
)



 df4 — Before
Mean AUC: 0.5971 ± 0.1076
              study      auc
             Dillon 0.634921
               Dinh 0.590476
            Dubourg 0.584516
      Lozupone 2013 0.506410
             Monaco 0.547591
     Noguera-Julian 0.500000
          Nowak2017 0.406863
      Pinto-Cardoso 0.616667
Serrano-Villar 2016 0.831481
Serrano-Villar 2017 0.678571
        Vesterbacka 0.519858
  Villanueva-Millan 0.642857
                 Yu 0.702484

 df4 — LatentGEE
Mean AUC: 0.5110 ± 0.1260
              study      auc
             Dillon 0.434524
               Dinh 0.587302
            Dubourg 0.398065
      Lozupone 2013 0.596154
             Monaco 0.547591
     Noguera-Julian 0.525887
          Nowak2017 0.462082
      Pinto-Cardoso 0.709091
Serrano-Villar 2016 0.614815
Serrano-Villar 2017 0.226190
        Vesterbacka 0.390071
  Villanueva-Millan 0.558036
                 Yu 0.593789

 df4 — ComBat
Mean AUC: 0.6355 ± 0.1337
              study      auc
             Dillon 0.811508
    

In [31]:
# LOSO study별 AUC 분산 비교
print("Before std:", auc_before['auc'].std())
print("LatentGEE std:", auc_latentgee['auc'].std())
print("ComBat std:", auc_combat['auc'].std())
print("MMUPHin std:", auc_mmuphin['auc'].std())
print("ConQuR std:", auc_conqur['auc'].std())

Before std: 0.10764449067558043
LatentGEE std: 0.125980753289138
ComBat std: 0.13371036442255899
MMUPHin std: 0.13717233185951924
ConQuR std: 0.10300511383551725


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd

def study_classification_auc(X, batch, label=""):
    X = np.array(X)
    batch = np.array(batch)
    
    le = LabelEncoder()
    batch_encoded = le.fit_transform(batch)
    n_classes = len(le.classes_)
    
    # 5-fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    
    for train_idx, test_idx in skf.split(X, batch_encoded):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = batch_encoded[train_idx], batch_encoded[test_idx]
        
        clf = OneVsRestClassifier(
            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
        )
        y_train_bin = label_binarize(y_train, classes=range(n_classes))
        clf.fit(X_train, y_train_bin)
        
        y_test_bin = label_binarize(y_test, classes=range(n_classes))
        y_prob = clf.predict_proba(X_test)
        
        auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
        aucs.append(auc)
    
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"(낮을수록 batch correction이 잘 된 것)")
    
    return {"label": label, "mean_auc": mean_auc, "std_auc": std_auc}

# 실행
batch_values = batch_filtered_df4['Study'].values

r_before   = study_classification_auc(X_filtered_df4.values, batch_values, "df4 — Before")
r_latentgee = study_classification_auc(X_df4_latgentgee.values, batch_values, "df4 — LatentGEE")
r_combat   = study_classification_auc(X_df4_combat.values, batch_values, "df4 — ComBat")
r_mmuphin  = study_classification_auc(X_df4_mmuphine.values, batch_values, "df4 — MMUPHin")
r_conqur   = study_classification_auc(X_df4_conqur.values, batch_values, "df4 — ConQuR")



 df4 — Before
Mean AUC: 0.9938 ± 0.0039
(낮을수록 batch correction이 잘 된 것)

 df4 — LatentGEE
Mean AUC: 0.6199 ± 0.0152
(낮을수록 batch correction이 잘 된 것)

 df4 — ComBat
Mean AUC: 1.0000 ± 0.0000
(낮을수록 batch correction이 잘 된 것)

 df4 — MMUPHin
Mean AUC: 0.9858 ± 0.0051
(낮을수록 batch correction이 잘 된 것)

 df4 — ConQuR
Mean AUC: 0.9991 ± 0.0009
(낮을수록 batch correction이 잘 된 것)


In [22]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

def hivstatus_classification_auc(X, y, label=""):
    X = np.array(X)
    y = np.array(y)
    
    # NaN 제거
    valid_mask = ~pd.isna(y)
    X = X[valid_mask]
    y = y[valid_mask]
    
    # label encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # 5-fold CV
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    
    for train_idx, test_idx in skf.split(X, y_encoded):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
        
        clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4)
        clf.fit(X_train, y_train)
        y_prob = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        aucs.append(auc)
    
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    print(f"\n{'='*50}")
    print(f" {label}")
    print(f"{'='*50}")
    print(f"Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    print(f"(높을수록 biological signal 보존이 잘 된 것)")
    
    return {"label": label, "mean_auc": mean_auc, "std_auc": std_auc}

# 실행
bio_labels = meta_filtered_df4['hivstatus'].values

r_before    = hivstatus_classification_auc(X_filtered_df4.values,          bio_labels, "df4 — Before")
r_latentgee = hivstatus_classification_auc(X_df4_latgentgee.values,   bio_labels, "df4 — LatentGEE")
r_combat    = hivstatus_classification_auc(X_df4_combat.values,   bio_labels, "df4 — ComBat")
r_mmuphin   = hivstatus_classification_auc(X_df4_mmuphine.values,  bio_labels, "df4 — MMUPHin")
r_conqur    = hivstatus_classification_auc(X_df4_conqur.values,   bio_labels, "df4 — ConQuR")


 df4 — Before
Mean AUC: 0.7225 ± 0.0235
(높을수록 biological signal 보존이 잘 된 것)

 df4 — LatentGEE
Mean AUC: 0.5087 ± 0.0368
(높을수록 biological signal 보존이 잘 된 것)

 df4 — ComBat
Mean AUC: 0.7542 ± 0.0201
(높을수록 biological signal 보존이 잘 된 것)

 df4 — MMUPHin
Mean AUC: 0.7271 ± 0.0139
(높을수록 biological signal 보존이 잘 된 것)

 df4 — ConQuR
Mean AUC: 0.9691 ± 0.0162
(높을수록 biological signal 보존이 잘 된 것)
